# Findings

### Flow of information
A common observation from the workshops was that climate information typically follows a chain of use within organizations. It begins with initial users—often from water resources or meteorology teams—who collect climate data and possess strong knowledge of its limitations and biases. These users process or reconstruct the data before passing it along to other teams, such as those responsible for load forecasting or outage planning. However, critical information about the data’s limitations is often lost along the way, as each team tends to assume that the data they receive has already been validated and corrected for uncertainties.

Another category of climate information users includes those who rely on standards, such as design values from organizations like the CSA or internal norms. These users may not be familiar with the origins or limitations of the data but trust that the authoritative body behind the standards ensures its reliability.

<div style="text-align: center;">
<img src="image/info_flow.png" alt="Workshop" width="800">
</div>

### Overview of decision-making challenges in the workshops

The Sankey diagram provides a preliminary overview of applications discussed during the workshops across participating organizations. It includes a limited set of applications and is not an exhaustive inventory.

The four columns connect electricity-sector activities, decision contexts, climate and environmental inputs, and input details and event types. Related decision contexts and input details are grouped to improve readability. Original labels remain available on hover and in the table. Connections with the same displayed endpoints are combined, with widths representing the sum of their recorded weights; hover details identify the contributing source records. Colours identify columns; the labels identify individual topics. Hover over a link to see its endpoints and recorded weight. On smaller screens, the diagram can be scrolled horizontally. The expandable table below provides all source records without requiring hover interaction. The radiation detail labelled "horizontal" and the original height entry "510m" are omitted from the diagram but retained in the table.

::: {.callout-note title="Interpreting the connections"}
Link widths reflect the weights recorded in the workshop synthesis, described as repeated citations in the initial draft. Their counting basis has not been documented sufficiently to interpret them as numbers of participants or quantitative importance. Totals differ between stages: these are associations, not a conserved flow. Connections through a shared node do not establish that an individual response followed a complete path across all four columns. A variable without an outgoing link has no characteristic recorded in this synthesis; this does not imply that no characteristic is relevant.
:::


In [1]:
#| echo: false

from collections import Counter
from html import escape
from pathlib import Path
import re
import textwrap

import yaml
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

# Keep every source record, including duplicate pairs and original weights.
with Path("sankey_nodes_links.yml").open(encoding="utf-8") as stream:
    data = yaml.safe_load(stream)

sections = ("sectors", "variables", "types")
column_titles = ("Electricity-sector activity", "Decision context", "Climate and environmental inputs", "Input details and event types")
palette = ("#245B85", "#227C78", "#9A6A24", "#64748B")
layers = [set() for _ in column_titles]
records = []
for stage, section in enumerate(sections):
    for item in data.get(section, []):
        source = item["source"]
        layers[stage].add(source)
        for entry in item.get("targets", []):
            target, value = entry if isinstance(entry, list) else (entry, 1)
            assert isinstance(value, (int, float)) and value > 0
            layers[stage + 1].add(target)
            records.append((stage, source, target, value))

# Presentation labels and the approved grouping do not modify source records.
raw_records = records[:]
display_labels = {
    (0, 'Generation Forecast'): 'Generation forecasting',
    (0, 'Generation Forecast - Hydro'): 'Generation forecasting',
    (0, 'Generation Forecast - Wind'): 'Generation forecasting',
    (0, 'Infra Design - Hydro'): 'Infrastructure design',
    (0, 'Infra Design - T&D'): 'Infrastructure design',
    (1, 'emergency plans'): 'Emergency and contingency planning',
    (1, 'extreme weather contingencies'): 'Emergency and contingency planning',
    (3, '5min'): 'Temporal resolution and aggregation',
    (3, '4hrs forecast'): '4-hour forecast',
    (3, '10m'): 'Measurement height',
    (3, '40m'): 'Measurement height',
    (3, '60m'): 'Measurement height',
    (3, '80m'): 'Measurement height',
    (3, '100m'): 'Measurement height',
    (1, 'load forecast'): 'Demand assessment',
    (1, 'flows<br>(historical and future)'): 'Streamflow assessment (historical and future)',
    (1, 'vulnerability assessment'): 'Risk and vulnerability assessment',
    (1, 'enterprise hazard<br>risk assessment'): 'Risk and vulnerability assessment',
    (1, 'new projects risk'): 'Risk and vulnerability assessment',
    (3, '30degC'): 'Temperature thresholds and derived indicators',
    (3, 'days < -35dgC'): 'Temperature thresholds and derived indicators',
    (3, 'hourly'): 'Temporal resolution and aggregation',
    (3, 'sub-daily'): 'Temporal resolution and aggregation',
    (3, 'daily'): 'Temporal resolution and aggregation',
    (3, 'monthly'): 'Temporal resolution and aggregation',
    (3, 'seasonal'): 'Temporal resolution and aggregation',
    (1, 'transmission line corridor'): 'Line and cable infrastructure design',
    (1, 'overhead lines'): 'Line and cable infrastructure design',
    (1, 'conductor design'): 'Line and cable infrastructure design',
    (1, 'underground system'): 'Line and cable infrastructure design',
    (1, 'peak demand'): 'Demand assessment',
    (1, 'peak-demand shift'): 'Demand assessment',
    (1, 'reliability outlook'): 'System reliability assessment',
    (1, 'system reliability'): 'System reliability assessment',
    (3, 'cooling degree days'): 'Temperature thresholds and derived indicators',
    (3, 'frost-free days'): 'Temperature thresholds and derived indicators',
    (3, 'growing season'): 'Temperature thresholds and derived indicators',
    (3, 'heating degree days'): 'Temperature thresholds and derived indicators',
    (3, 'high temperature & low wind'): 'Combined conditions',
    (3, 'ice & wind'): 'Combined conditions',
    (3, 'snow & wind'): 'Combined conditions',
    (3, 'direction'): 'Wind characteristics',
    (3, 'speed'): 'Wind characteristics',
    (1, 'PMP / PMF / IDF'): 'Hydrological design criteria',
    (1, 'hydraulic asset refurbishment'): 'Generation infrastructure design and refurbishment',
    (1, 'power plant design'): 'Generation infrastructure design and refurbishment',
    (1, 'improve inspection'): 'Inspection planning',
    (1, 'equipment rating'): 'Equipment rating assessment',
    (1, 'outages process'): 'Outage planning',
    (2, 'flows'): 'Streamflow',
    (3, 'dry bulb'): 'Dry-bulb temperature',
    (3, 'dewpoint'): 'Dew-point temperature',
    (1, 'energy & revenue'): 'Energy production and revenue assessment',
    (3, 'extreme / tail distribution'): 'Extremes and distribution tails',
    (3, 'freeze and thaw'): 'Freeze–thaw conditions',
    (2, 'humidity'): 'Humidity and moisture',
    (2, 'moisture'): 'Humidity and moisture',
    (1, 'drought impact'): 'Drought impact assessment',
    (1, 'GHG emission'): 'GHG emissions assessment',
    (1, 'facilitate adaptation'): 'Adaptation planning and support',
}

def sentence_case(label):
    # Display formatting only: preserve technical acronyms and temperature units.
    text = label.lower()
    text = re.sub(r"\b(?:ESG|GHG|T&D|PMP|PMF|IDF)\b",
                  lambda match: match.group().upper(), text, flags=re.I)
    text = text.replace("\u00b0c", "\u00b0C")
    return text[:1].upper() + text[1:]

def diagram_label(stage, label):
    return sentence_case(display_labels.get((stage, label), label))

# Approved omissions apply to the diagram only; the table retains source records.
omitted_details = {"horizontal", "510m"}
diagram_records = [record for record in raw_records
                   if not (record[0] == 2 and record[2] in omitted_details)]
records = [(stage, diagram_label(stage, source), diagram_label(stage + 1, target), value)
           for stage, source, target, value in diagram_records]
layers = [set() for _ in column_titles]
node_members = {}
for original, mapped in zip(diagram_records, records):
    stage, source, target, _ = mapped
    layers[stage].add(source)
    layers[stage + 1].add(target)
    node_members.setdefault((stage, source), set()).add(original[1])
    node_members.setdefault((stage + 1, target), set()).add(original[2])

# Consolidate parallel displayed links, retaining every contributing source row.
link_totals = Counter()
link_members = {}
for original, (stage, source, target, value) in zip(diagram_records, records):
    key = (stage, source, target)
    link_totals[key] += value
    link_members.setdefault(key, []).append(original)
records = [(stage, source, target, value)
           for (stage, source, target), value in link_totals.items()]

# Layer-qualified identifiers prevent accidental merging of equal labels.
groups = [sorted(layer) for layer in layers]

def crossing_score(order):
    """Unweighted crossing count; weights used for drawing remain unchanged."""
    ranks = [{label: i for i, label in enumerate(group)} for group in order]
    score = 0
    for stage in range(len(sections)):
        edges = [(ranks[stage][s], ranks[stage + 1][t])
                 for st, s, t, _ in records if st == stage]
        score += sum((a - c) * (b - d) < 0
                     for i, (a, b) in enumerate(edges) for c, d in edges[i + 1:])
    return score

# Deterministic weighted-barycentre sweeps; retain the best complete ordering.
alphabetical_crossings = crossing_score(groups)
best = [group[:] for group in groups]
best_score = alphabetical_crossings
for _ in range(8):
    for direction in (1, -1):
        indices = range(1, 4) if direction == 1 else range(2, -1, -1)
        for stage in indices:
            neighbour = stage - direction
            ranks = {label: i / max(1, len(groups[neighbour]) - 1)
                     for i, label in enumerate(groups[neighbour])}
            previous = {label: i for i, label in enumerate(groups[stage])}
            group_length = len(groups[stage])
            def barycentre(label):
                adjacent = [(s, v) for st, s, t, v in records
                            if st == neighbour and t == label] if direction == 1 else [
                            (t, v) for st, s, t, v in records if st == stage and s == label]
                if not adjacent:
                    return previous[label] / max(1, group_length - 1)
                return sum(ranks[n] * v for n, v in adjacent) / sum(v for _, v in adjacent)
            groups[stage].sort(key=lambda label: (barycentre(label), previous[label]))
        score = crossing_score(groups)
        if score < best_score:
            best, best_score = [group[:] for group in groups], score
groups = best

def plain_label(label):
    return re.sub(r"<br\s*/?>", " ", label, flags=re.I)

def wrapped_label(label):
    return "<br>".join(escape(line) for line in textwrap.wrap(plain_label(label), width=28))

nodes = [(stage, label) for stage, group in enumerate(groups) for label in group]
index = {node: i for i, node in enumerate(nodes)}
incoming, outgoing = Counter(), Counter()
for stage, source, target, value in records:
    outgoing[(stage, source)] += value
    incoming[(stage + 1, target)] += value
node_values = {node: max(incoming[node], outgoing[node]) for node in nodes}

height, top_margin, bottom_margin, padding = 1544, 120, 24, 28
plot_height = height - top_margin - bottom_margin
scale = min((plot_height - (len(group) - 1) * padding) /
            sum(node_values[(stage, label)] for label in group)
            for stage, group in enumerate(groups))
x_columns = (0.005, 0.29, 0.62, 0.92)
positions = {}
for stage, group in enumerate(groups):
    total_height = sum(node_values[(stage, label)] * scale for label in group)
    gap = (plot_height - total_height) / max(1, len(group) - 1)
    cursor = 0
    for label in group:
        node_height = node_values[(stage, label)] * scale
        positions[(stage, label)] = (cursor + node_height / 2) / plot_height
        cursor += node_height + gap

def translucent(hex_colour, alpha):
    rgb = tuple(int(hex_colour[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})"

fig = go.Figure(go.Sankey(
    arrangement="fixed",
    node=dict(
        label=[wrapped_label(label) for _, label in nodes],
        color=[palette[stage] for stage, _ in nodes],
        x=[x_columns[stage] for stage, _ in nodes],
        y=[positions[node] for node in nodes],
        pad=padding, thickness=15,
        line=dict(color="rgba(35,49,62,0.55)", width=0.5),
        customdata=[[column_titles[stage], escape(plain_label(label)),
                     "<br>".join(escape(plain_label(member)) for member in sorted(node_members[(stage, label)]))]
                    for stage, label in nodes],
        hovertemplate="%{customdata[0]}<br><b>%{customdata[1]}</b><br>Original labels:<br>%{customdata[2]}<extra></extra>",
    ),
    link=dict(
        source=[index[(stage, source)] for stage, source, _, _ in records],
        target=[index[(stage + 1, target)] for stage, _, target, _ in records],
        value=[value for _, _, _, value in records],
        color=[translucent(palette[stage], 0.18) for stage, _, _, _ in records],
        hovercolor=[translucent(palette[stage], 0.65) for stage, _, _, _ in records],
        customdata=[[escape(plain_label(source)), escape(plain_label(target)),
                     f"{column_titles[stage]} → {column_titles[stage + 1]}",
                     "<br>".join(
                         f"{escape(plain_label(original[1]))} → {escape(plain_label(original[2]))} (weight {original[3]})"
                         for original in link_members[(stage, source, target)]),
                     len(link_members[(stage, source, target)])]
                    for stage, source, target, _ in records],
        hovertemplate=("%{customdata[2]}<br><b>%{customdata[0]} → %{customdata[1]}</b>"
                       "<br>Sum of recorded weights: %{value}<br>Source records: %{customdata[4]}"
                       "<br>Contributing connections:<br>%{customdata[3]}<extra></extra>"),
    ),
))
fig.update_layout(
    height=height, autosize=True,
    margin=dict(l=24, r=24, t=top_margin, b=bottom_margin),
    font=dict(family="Lato, Arial, sans-serif", size=16, color="#23313E"),
    hoverlabel=dict(font_size=14, bgcolor="white", font_color="#23313E"),
    paper_bgcolor="white", plot_bgcolor="white",
    annotations=[dict(x=x, y=1.06, xref="paper", yref="paper", text="<b>" + "<br>".join(escape(line) for line in textwrap.wrap(title, width=24)) + "</b>",
                      showarrow=False, xanchor=("left" if i == 0 else "right" if i == 3 else "center"),
                      font=dict(size=19, color=palette[i]))
                 for i, (title, x) in enumerate(zip(column_titles, x_columns))],
)

# Retain the existing Plotly CDN delivery; no new supporting files are needed.
chart = pio.to_html(fig, include_plotlyjs="cdn", full_html=False,
                    div_id="workshop-sankey", config=dict(
                        responsive=True, displaylogo=False,
                        modeBarButtonsToRemove=["select2d", "lasso2d"],
                        toImageButtonOptions=dict(filename="workshop-climate-data-connections")))
display(HTML(f'''<div role="region" aria-label="Workshop connections diagram; scroll horizontally on narrow screens"
tabindex="0" style="overflow-x:auto; max-width:100%; border:1px solid #dce3e8; border-radius:4px;">
<div style="min-width:1050px;">{chart}</div></div>'''))

# A keyboard-accessible alternative retains all records, including repeated pairs.
rows = "".join(f"<tr><td>{escape(column_titles[stage])} → {escape(column_titles[stage + 1])}</td>"
               f"<td>{escape(plain_label(source))}</td><td>{escape(plain_label(target))}</td>"
               f"<td>{value}</td></tr>" for stage, source, target, value in raw_records)
display(HTML(f'''<details style="margin-top:1rem;"><summary>View all recorded connections ({len(records)} link records)</summary>
<p>Original labels and weights are retained in this table, including labels grouped, reformatted, or omitted from the diagram. Repeated source–target pairs remain separate records.</p>
<div tabindex="0" role="region" aria-label="Recorded workshop connections table" style="overflow:auto; max-height:32rem;">
<table class="table table-striped table-sm"><caption>Workshop synthesis: recorded pairwise connections</caption>
<thead><tr><th scope="col">Stage</th><th scope="col">Source</th><th scope="col">Target</th><th scope="col">Recorded weight</th></tr></thead>
<tbody>{rows}</tbody></table></div></details>'''))


Stage,Source,Target,Recorded weight
Electricity-sector activity → Decision context,Capacity Expansion Modelling,long term planning,1
Electricity-sector activity → Decision context,Demand Forecasting,load forecast,2
Electricity-sector activity → Decision context,Demand Forecasting,peak demand,2
Electricity-sector activity → Decision context,Electrical System Operations,mitigation and adaptation indigenous communities,1
Electricity-sector activity → Decision context,Electrical System Operations,extreme weather contingencies,1
Electricity-sector activity → Decision context,Electrical System Operations,equipment rating,1
Electricity-sector activity → Decision context,Electrical System Operations,outages process,1
Electricity-sector activity → Decision context,Electrical System Operations,reliability outlook,1
Electricity-sector activity → Decision context,ESG/Sustainability Reporting,GHG emission,3
Electricity-sector activity → Decision context,ESG/Sustainability Reporting,peak-demand shift,1


The workshops highlighted a diverse range of decision-making challenges across various sectors, each with unique climate data needs and considerations. However, several gaps in available climate datasets emerged:

 - **Combined events**: Many applications require understanding the interplay of multiple climate variables. For instance, transmission and distribution of overhead lines are affected by the combined effects of wind and ice accumulation. CSA standard consider the combination of 1/50 years wind and ice loads for design purposes. Freezing rain as well as wet snow are also of interest for this sector as it affects component strength ratings and even the consideration of using underground in rural areas. Vegetation management also takes an interest in these combined events as they can lead to tree falls and power outages.

 - **Extremes**: A significant number of applications focus on extreme weather events of climate variables. Most infrastructure designs are made for rare events (e.g., 1/50 years, 1/100 years, even 1/1000) to assure the structure resilience. If they are not directly included in standards (such as precipitation, wind, temperature and ice), they can be estimated from climate dataset and IDF curves for precipitation. Many other sectors look at extremes to plan and assure system reliability such as demand forecasting that looks at extreme temperature events to evaluate peak demand.

 - **High impact events**: High impact events refer to natural disaster such as droughts, floods, wildfires, or storms. These are of particular concern for enterprise risk assessment and emergency planning. Understanding the frequency, intensity, and potential impacts of such events are crucial for developing effective mitigation and response strategies. They are also relevant for all sectors involved in infrastructure planning and operation to ensure resilience against such events.

 - **Hourly data**: Several decision-making contexts require high-resolution temporal data (hourly or sub-hourly) to capture the dynamics of weather events and demand. Hourly wind data is particularly critical for wind power generation forecasting and outages management. Hydroelectricity generation forecasting also require to use sub-daily dataset to have a better predictor of energy. Similarly, demand forecasting also plans for hourly peaks in electricity consumption and therefore even econometric models are often built on an hourly basis. Hourly data is also used for HVAC system design (weather files). Professionals working in hydraulic design, dam safety review, or streamflow projections were also adamant about generally needing higher resolution data, equally temporal or spatial. 

 - **Urban vs. rural data**: Some applications distinguish between urban and rural settings, recognizing that climate impacts can vary significantly based on location, especially for temperature and wind. This is particularly relevant for transmission and distribution infrastructure design where urban heat island can affect lines sagging as well as conductor ratings for urban stations. Wind speed can also affect overhead lines differently in urban vs. rural areas due to building interference versus open areas. 
